In [1]:
import os
import pandas as pd
import numpy as np
import datetime as dt
from preprocessing import Preprocessing
import pickle
from tqdm import tqdm

# silence warnings
import warnings
warnings.filterwarnings('ignore')

In [2]:
print(f'Latest run date: {dt.datetime.today()}')

Latest run date: 2025-02-10 21:19:08.433928


#### Functions

In [3]:
def convert_list_none_to_nan(list_vals):
    try:
        list_out = [np.nan if str_val is None else str_val for str_val in list_vals]
    except TypeError:
        list_out = np.nan
    return list_out

#### Constants

In [4]:
# project
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

# task
str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

str_dirname_output = './output'

# quantile for capping income
flt_quantile = 0.99

Project: 20241112-simple-model-test
Task: 08_prep_data


#### Make output directory


In [5]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

#### Import data

In [6]:
%%time

str_filename = 'df.gzip'
str_uri = f's3://{str_project}/07_join_targets/{str_filename}'
df = pd.read_parquet(
    str_uri,
)
# sort
df.sort_values(by='applicationdate__app', ascending=True, inplace=True)
# show
df

CPU times: user 9.82 s, sys: 3.71 s, total: 13.5 s
Wall time: 5.67 s


,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,Early_Pay_Delinquency_30_90_Flag,Early_Pay_Delinquency_30_180_Flag,Early_Pay_Delinquency_30_360_Flag,Early_Pay_Delinquency_60_720_Flag,run_date,days_on_books,loss_at_60,loss_at_180,loss_at_360,loss_at_720
37865,5514485,2022-10-15 02:35:52.2455863,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Virginia,Independent,Virginia,False,...,0,0,0,0,2025-01-31 10:43:13.940299,1467,0.0,0.0,0.0,0.0
37866,5514970,2022-10-15 02:36:32.6396499,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,California,Franchise,California,False,...,0,0,0,0,2025-01-31 10:43:13.940299,1464,0.0,0.0,0.0,0.0
37869,5515245,2022-10-15 02:36:53.5525392,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Texas,Franchise,Texas,False,...,0,0,0,0,2025-01-31 10:43:13.940299,1465,0.0,0.0,0.0,0.0
37870,5515340,2022-10-15 02:37:13.5377961,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Arizona,Franchise,Arizona,False,...,0,0,0,0,2025-01-31 10:43:13.940299,1453,0.0,0.0,0.0,0.0
37871,5515580,2022-10-15 02:37:34.1144830,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,0,0,Kentucky,Franchise,Kentucky,False,...,0,0,0,0,2025-01-31 10:43:13.940299,1464,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94425,8420461,2024-11-26 02:27:09+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Pennsylvania,Independent,Pennsylvania,True,...,0,0,0,0,2025-01-31 10:43:13.940299,67,0.0,0.0,0.0,0.0
94472,8420588,2024-11-26 06:16:16+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,North Carolina,Independent,North Carolina,True,...,0,0,0,0,2025-01-31 10:43:13.940299,67,0.0,0.0,0.0,0.0
94426,8420665,2024-11-26 02:37:11+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Ohio,Franchise,Ohio,True,...,0,0,0,0,2025-01-31 10:43:13.940299,67,0.0,0.0,0.0,0.0
94456,8421889,2024-11-26 05:10:18+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Georgia,Franchise,Georgia,True,...,0,0,0,0,2025-01-31 10:43:13.940299,67,0.0,0.0,0.0,0.0


#### Prep flt_payment__tu_pmthx for preprocessing

In [7]:
# df['flt_payment__tu_pmthx'] = df['flt_payment__tu_pmthx'].str.replace('nan', 'None')
# df['flt_payment__tu_pmthx'] = df['flt_payment__tu_pmthx'].apply(eval)
# # make None into np.nan cause thats how it will be in prod
# df['flt_payment__tu_pmthx'] = df['flt_payment__tu_pmthx'].apply(
#     lambda x: x if isinstance(x, list) else np.nan if x is None else x,
# )
# # in each list replace with np.nan cause thats how it will be in prod
# df['flt_payment__tu_pmthx'] = df['flt_payment__tu_pmthx'].apply(convert_list_none_to_nan)

#### Initialize class

In [8]:
cls_model_preprocessing = Preprocessing(
    dict_impute={}, # placeholder for now
    dict_bins={}, # placeholder for now
    flt_quantile=flt_quantile,
    int_new_payment=600,
)

#### Fit

In [9]:
cls_model_preprocessing.fit(
    X=df,
)

Getting max income based on 0.99 quantile...
Max Income: 11742.160000000047


#### Save

In [10]:
str_filename = 'cls_model_preprocessing.pkl'
str_local_path = f'{str_dirname_output}/{str_filename}'
pickle.dump(cls_model_preprocessing, open(str_local_path, 'wb'))

#### Transform

In [11]:
df = cls_model_preprocessing.transform(
    X=df,
)

# show
df

Masking negative values to NaN...


100%|██████████| 2113/2113 [00:03<00:00, 664.89it/s]


Capping income...
Replacing zeros...


100%|██████████| 3/3 [00:00<00:00, 923.86it/s]


Engineering number of months...
Engineering number of months total...
Engineering weighted average...
Engineering tag for has auto...
Engineering tag for open auto indicator...
Engineering tag for closed auto indicator...
Engineering tag for open and closed auto indicator...
Engineering 3 month early delinquency...
Engineering 6 month early delinquency...
Engineering 3 month recent delinquency...
Engineering 6 month recent delinquency...
Engineering DTI...
Engineering franchise...
Engineering has a codebtor...
Engineering vehicle age...
Engineering PTI...
Engineering LTV...
Engineering BK...
Engineering perfect payment history tag for most recent auto...
Engineering perfect payment history tag for open auto...
Engineering perfect payment history tag for closed auto...
Engineering interactions...
Imputing values...


0it [00:00, ?it/s]


Binning values for scorecard...


0it [00:00, ?it/s]


,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,ENG-franchise,ENG-has_codebtor,ENG-vehicle_age,ENG-payment_to_income,ENG-loan_to_value,ENG-bk,ENG-perfect_payment_hx,ENG-perfect_payment_hx_open,ENG-perfect_payment_hx_closed,ENG-bk_x_wtd_avg
37865,5514485,2022-10-15 02:35:52.2455863,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Virginia,Independent,Virginia,False,...,0,0,5,0.111662,1.299997,0,1,1,0,0.000000
37866,5514970,2022-10-15 02:36:32.6396499,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,California,Franchise,California,False,...,1,0,6,0.055245,1.295059,0,0,0,0,0.000000
37869,5515245,2022-10-15 02:36:53.5525392,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Texas,Franchise,Texas,False,...,1,0,0,0.087712,1.156227,0,0,0,0,0.000000
37870,5515340,2022-10-15 02:37:13.5377961,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Arizona,Franchise,Arizona,False,...,1,0,5,NaN,1.599743,0,0,0,0,NaN
37871,5515580,2022-10-15 02:37:34.1144830,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,0,0,Kentucky,Franchise,Kentucky,False,...,1,1,4,NaN,1.363642,0,1,0,1,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94425,8420461,2024-11-26 02:27:09+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Pennsylvania,Independent,Pennsylvania,True,...,0,0,5,NaN,1.585118,0,0,0,0,0.000000
94472,8420588,2024-11-26 06:16:16+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,North Carolina,Independent,North Carolina,True,...,0,0,4,NaN,1.280957,0,0,0,0,0.000000
94426,8420665,2024-11-26 02:37:11+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Ohio,Franchise,Ohio,True,...,1,0,2,NaN,1.189153,1,0,0,0,0.136585
94456,8421889,2024-11-26 05:10:18+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Georgia,Franchise,Georgia,True,...,1,0,0,0.206963,1.082309,0,0,0,1,0.000000


#### Save to s3

In [12]:
%%time

str_filename = 'df.gzip'
str_uri = f's3://{str_project}/{str_task}/{str_filename}'
df.to_parquet(str_uri, compression='gzip')

CPU times: user 32.3 s, sys: 272 ms, total: 32.6 s
Wall time: 32.3 s
